# Downloading the dataset

Builds the street-object dataset for `training.ipynb` straight from the COCO website.

## 1. Dependencies

This cell installs and checks if all necessary dependencies are installed.
If not execute in the root directory ``pip install -r requirments.txt``

In [ ]:
import importlib.util

REQUIRED = ["yaml"]

missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing:", " ".join(missing))
else:
    print("All dependencies installed.")

import json
import time
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import yaml

print("everything else comes from the standard library")

## 2. Setup

Locate the repo root so the dataset lands in ``datasets/`` next to the source, no matter
which directory the notebook is started from.

In [ ]:
def repo_root(start: Path | None = None) -> Path:
    """Walk up from the notebook until the directory holding the model package appears."""
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "src" / "model.py").exists() or (cand / "model.py").exists():
            return cand
    raise FileNotFoundError(f"no repo root above {here}")


ROOT = repo_root()
DATASETS = ROOT / "datasets"
DATASETS.mkdir(exist_ok=True)

print("repo root:", ROOT)
print("datasets: ", DATASETS)

## 3. Download settings

`CLASSES` are the six street objects the model is trained on. Their order fixes the class
indices in the label files (`person -> 0` ... `truck -> 5`), so keep it in sync with the
`names` of any checkpoint you want to keep using.

`MAX_SAMPLES` caps the images **per split** (the first N by file name). All of it is
~70k train and ~3k val images, about 11 GB. Set
it to e.g. `2000` for a quick pipeline test, `None` for everything.

Everything is resumable, images already on disk are skipped, so a canceled run just
continues where it stopped when you re-execute the cells.

In [ ]:
# --- CHANGE ME -----------------------------------------------------------------------
NAME = "coco_street_objects_yolo"
CLASSES = ["person", "bicycle", "motorcycle", "car", "bus", "truck"]
MAX_SAMPLES = None   # None -> every image containing one of the classes
WORKERS = 32         # parallel image downloads; 64 helps on a fast line
# -------------------------------------------------------------------------------------

EXPORT_DIR = DATASETS / NAME
CACHE = DATASETS / ".coco_cache"
CACHE.mkdir(exist_ok=True)

SPLITS = {"train": "train2017", "val": "val2017"}
ANN_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMG_URL = "http://images.cocodataset.org/{coco_split}/{file_name}"

print("export dir:", EXPORT_DIR)
print("classes:   ", dict(enumerate(CLASSES)))
print("per split: ", MAX_SAMPLES or "all")

## 4. Annotations

One zip with the boxes of both official COCO splits.

In [ ]:
def download(url, dest, attempts=10):
    """Resumable download with retries. Returns immediately if dest is already complete."""
    dest = Path(dest)
    total = None
    try:
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30) as r:
            total = int(r.headers["Content-Length"])
    except Exception:
        pass
    if total and dest.exists() and dest.stat().st_size == total:
        print(f"{dest.name}: already downloaded ({total / 1e6:.0f} MB)")
        return dest

    for attempt in range(1, attempts + 1):
        have = dest.stat().st_size if dest.exists() else 0
        if total and have >= total:
            return dest
        req = urllib.request.Request(url)
        if have:
            req.add_header("Range", f"bytes={have}-")
        try:
            with urllib.request.urlopen(req, timeout=60) as r, open(dest, "ab" if have else "wb") as f:
                t0, done = time.time(), have
                while chunk := r.read(1 << 20):
                    f.write(chunk)
                    done += len(chunk)
                    speed = (done - have) / 1e6 / max(time.time() - t0, 1e-9)
                    pct = f"{100 * done / total:5.1f}%" if total else "  ?  "
                    print(f"\r{dest.name}: {pct}  {done / 1e6:6.0f} MB  {speed:5.1f} MB/s", end="")
            print()
            if total is None or dest.stat().st_size >= total:
                return dest
        except Exception as exc:
            print(f"\n[{dest.name} attempt {attempt}] {exc} -- resuming in 5s")
            time.sleep(5)
    raise RuntimeError(f"download of {url} kept failing after {attempts} attempts")


ANN_ZIP = download(ANN_URL, CACHE / "annotations_trainval2017.zip")

with zipfile.ZipFile(ANN_ZIP) as z:
    print([i.filename for i in z.infolist() if i.filename.startswith("annotations/instances")])

## 5. Labels

Turns the COCO json into YOLO label files: one `.txt` per image, one line per box with
`class cx cy w h` normalized to the image size.

In [ ]:
def yolo_labels(coco_split):
    """{file_name: [label lines]} for every image holding >= 1 box of CLASSES."""
    with zipfile.ZipFile(ANN_ZIP) as z, z.open(f"annotations/instances_{coco_split}.json") as f:
        data = json.load(f)

    cat_to_idx = {c["id"]: CLASSES.index(c["name"]) for c in data["categories"] if c["name"] in CLASSES}
    images = {im["id"]: im for im in data["images"]}

    out = {}
    for a in data["annotations"]:
        idx = cat_to_idx.get(a["category_id"])
        if idx is None or a.get("iscrowd"):
            continue
        im = images[a["image_id"]]
        iw, ih = im["width"], im["height"]
        x, y, w, h = a["bbox"]
        x2, y2 = min(x + w, iw), min(y + h, ih)     # clip to the image
        x, y = max(x, 0.0), max(y, 0.0)
        w, h = x2 - x, y2 - y
        if w <= 1 or h <= 1:                        # degenerate box
            continue
        out.setdefault(im["file_name"], []).append(
            f"{idx} {(x + w / 2) / iw:.6f} {(y + h / 2) / ih:.6f} {w / iw:.6f} {h / ih:.6f}"
        )

    del data, images
    return dict(sorted(out.items()))


labels = {}
for split, coco_split in SPLITS.items():
    t0 = time.time()
    found = yolo_labels(coco_split)
    labels[split] = dict(list(found.items())[:MAX_SAMPLES]) if MAX_SAMPLES else found

    out_dir = EXPORT_DIR / "labels" / split
    out_dir.mkdir(parents=True, exist_ok=True)
    for file_name, lines in labels[split].items():
        (out_dir / f"{Path(file_name).stem}.txt").write_text("\n".join(lines) + "\n")

    n_box = sum(len(v) for v in labels[split].values())
    kept = f"{len(labels[split])} of {len(found)}" if MAX_SAMPLES else str(len(found))
    print(f"{split:5s} {kept} images  {n_box} boxes  ({time.time() - t0:.0f}s)")

for split, per_image in labels.items():
    counts = {c: 0 for c in CLASSES}
    for lines in per_image.values():
        for line in lines:
            counts[CLASSES[int(line.split()[0])]] += 1
    print(f"\n{split} boxes per class:")
    for c, n in counts.items():
        print(f"  {c:12s} {n:7d}")

## 6. Images

Downloads exactly the images listed above, straight into the YOLO folder layout.

Files already on disk are skipped, so re-running this cell only fetches what is missing.

In [ ]:
def fetch(job):
    """Download one image. Returns its size, 0 if it was already there, -1 on failure."""
    coco_split, file_name, dest = job
    if dest.exists() and dest.stat().st_size:
        return 0
    url = IMG_URL.format(coco_split=coco_split, file_name=file_name)
    for _ in range(5):
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                blob = r.read()
            tmp = dest.with_suffix(".part")
            tmp.write_bytes(blob)
            tmp.rename(dest)          # rename is atomic -> no half files on interrupt
            return len(blob)
        except Exception:
            time.sleep(1)
    return -1


for split, per_image in labels.items():
    out_dir = EXPORT_DIR / "images" / split
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(SPLITS[split], name, out_dir / name) for name in per_image]

    t0, got, skipped, failed = time.time(), 0, 0, []
    with ThreadPoolExecutor(WORKERS) as pool:
        for i, (job, size) in enumerate(zip(jobs, pool.map(fetch, jobs)), 1):
            if size < 0:
                failed.append(job[1])
            elif size == 0:
                skipped += 1
            else:
                got += size
            if i % 200 == 0 or i == len(jobs):
                el = max(time.time() - t0, 1e-9)
                eta = (len(jobs) - i) / (i / el) / 60
                print(f"\r{split}: {i}/{len(jobs)}  {got / 1e9:5.2f} GB  {got / 1e6 / el:4.1f} MB/s"
                      f"  {i / el:4.0f} img/s  eta {eta:5.1f} min  skipped {skipped}  failed {len(failed)}", end="")
    print()
    if failed:
        print(f"  {len(failed)} images failed, e.g. {failed[:3]} -- re-run this cell to retry them")

## 7. dataset.yaml

The descriptor Ultralytics reads. `path` is written absolute so the dataset can be
trained on from any working directory.

In [ ]:
DATA = EXPORT_DIR / "dataset.yaml"
DATA.write_text(yaml.safe_dump({
    "path": str(EXPORT_DIR),
    "train": "images/train",
    "val": "images/val",
    "names": dict(enumerate(CLASSES)),
}, sort_keys=False))

print(DATA)
print()
print(DATA.read_text())

## 8. Check the export

Every image needs a label file of the same name, so the two counts per split should
match. The annotation zip in `datasets/.coco_cache/` is only needed to rebuild the
labels and can be deleted afterwards.

In [ ]:
for split in SPLITS:
    n_img = sum(1 for _ in (EXPORT_DIR / "images" / split).glob("*.jpg"))
    n_lbl = sum(1 for _ in (EXPORT_DIR / "labels" / split).glob("*.txt"))
    print(f"{split:5s} {n_img:6d} images  {n_lbl:6d} labels  ({'ok' if n_img == n_lbl else 'MISMATCH'})")

size = sum(p.stat().st_size for p in EXPORT_DIR.rglob("*") if p.is_file())
print(f"\ndataset: {size / 1e9:.2f} GB in {EXPORT_DIR}")
print(f"cache:   {sum(p.stat().st_size for p in CACHE.glob('*')) / 1e9:.2f} GB in {CACHE}")


from ultralytics.data.utils import check_det_dataset

check_det_dataset(str(DATA))
print("\ndataset.yaml accepted by ultralytics")

## Next step

Open `training.ipynb` and point its `DATA` variable at the yaml printed above:

```python
DATA = Path(r"<repo>/datasets/coco_street_objects_yolo/dataset.yaml")
```